# Public Data Access

Explore public nmrXiv projects, samples, and datasets. These endpoints do not require login.

In [1]:
import os
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

In [2]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def as_dataframe(payload):
    if isinstance(payload, list):
        return pd.json_normalize(payload)
    if isinstance(payload, dict):
        for key in ["data", "projects", "samples", "datasets", "items", "results"]:
            if isinstance(payload.get(key), list):
                return pd.json_normalize(payload[key])
        return pd.json_normalize(payload)
    return pd.DataFrame({"value": [payload]})


def clean_params(params):
    return {key: value for key, value in params.items() if value not in (None, "")}


def list_models(
    model,
    per_page=100,
    page=1,
    sort="-created_at",
    name=None,
    identifier=None,
    doi=None,
    created_at=None,
):
    if model not in {"projects", "samples", "datasets"}:
        raise ValueError("model must be one of: projects, samples, datasets")
    params = clean_params({
        "per_page": per_page,
        "page": page,
        "sort": sort,
        "filter[name]": name,
        "filter[identifier]": identifier,
        "filter[doi]": doi,
        "filter[created_at]": created_at,
    })
    return api_request("GET", f"/v1/list/{model}", params=params)


def get_model(public_id):
    return api_request("GET", f"/v1/{public_id}")

## Retrieve public scientific data collections

Fetches paginated collections of publicly available scientific data models from the nmrXiv repository. Supports projects, samples, and datasets.

Documented query parameters:

- `per_page`: number of results per page, default `100`, maximum `500`
- `page`: page number for pagination, default `1`
- `sort`: one of `created_at`, `-created_at`, `identifier`, `-identifier`
- `filter[name]`: case-insensitive partial match on name/title
- `filter[identifier]`: exact NMRXIV identifier match, e.g. `P123`
- `filter[doi]`: DOI match
- `filter[created_at]`: ISO date or date range, e.g. `2024-01-01,2024-12-31`

In [3]:
projects = list_models(
    model="projects",
    per_page=100,
    page=1,
    sort="-created_at",
)
projects_df = as_dataframe(projects)
projects_df.shape

GET https://dev.nmrxiv.org/api/v1/list/projects?per_page=100&page=1&sort=-created_at -> 200


(9, 33)

## Filter examples

Uncomment or edit values to combine filters. Query parameter names are mapped by the `list_models(...)` helper.

In [4]:
filtered_projects = list_models(
    model="projects",
    per_page=20,
    page=1,
    sort="identifier",
    name="Project-test-Embargo",
    identifier=None,
    doi=None,
    created_at=None,
)

filtered_projects_df = as_dataframe(filtered_projects)
filtered_projects_df.head()

GET https://dev.nmrxiv.org/api/v1/list/projects?per_page=20&page=1&sort=identifier&filter%5Bname%5D=Project-test-Embargo -> 200


,id,name,slug,description,photo_url,is_public,is_published,identifier,schema_version,public_url,...,owner.profile_photo_url,stats.likes,license.title,license.slug,license.spdx_id,license.url,license.html_url,license.permissions,license.description,license.body
0,27,Project-test-Embargo,project-test-embargo,"Lorem ipsum dolor sit amet, consectetur adipis...",,True,True,NMRXIV:P971,beta,https://dev.nmrxiv.org/project/P971,...,https://ui-avatars.com/api/?name=Super+admin%2...,0,Community Data License Agreement Permissive 1....,cdla-permissive-1.0,CDLA-Permissive-1.0,https://cdla.dev/permissive-1-0/,None,None,This is the Community Data License Agreement –...,<b>Definitions</b><br><br>1.1 “Add” means to s...


In [ ]:
project_by_identifier = list_models(
    model="projects",
    per_page=10,
    page=1,
    sort="-created_at",
    identifier="P16",
)

as_dataframe(project_by_identifier).head()

In [ ]:
projects_created_in_range = list_models(
    model="projects",
    per_page=20,
    page=1,
    sort="-created_at",
    created_at="2024-01-01,2024-12-31",
)

as_dataframe(projects_created_in_range).head()

In [ ]:
projects_df.head()

## List samples and datasets

In [ ]:
samples_df = as_dataframe(list_models("samples"))
datasets_df = as_dataframe(list_models("datasets"))

samples_df.shape, datasets_df.shape

## Retrieve specific public scientific data by identifier​

Fetches detailed information for a specific publicly available scientific data entry using its NMRXIV identifier. Returns comprehensive metadata, associated files, measurement details, and structured data compliant with scientific data standards. Supports projects (P prefix), samples (S prefix), and datasets (D prefix). Use identifiers such as `P16`, `S70`, or `D399`.

In [ ]:
public_id = "P16"
model_detail = get_model(public_id)
model_detail

## Helpful columns

In [ ]:
columns = [col for col in ["identifier", "name", "slug", "description", "is_public", "is_published"] if col in projects_df.columns]
projects_df[columns].head(20)